# 00 · Prepare the data

**What this step does.** Downloads the three datasets (Diabetes 130, CDC Diabetes Health Indicators, Taiwan credit-card default), cleans them, and splits each one **once** into train (60%), validation (20%) and test (20%).

**Why it matters.**
- Every later notebook reads these files, so every result traces back to one fixed split.
- The **test set is never corrupted.** It is the ground truth for every score in the paper.
- Cleaning removes records whose labels are not real observations:
  - *Diabetes 130:* patients who died or went to hospice cannot be readmitted, so their "not readmitted" label is set by the discharge, not observed. They are dropped.
  - *Diabetes 130:* patients appear more than once. Keeping one encounter per patient stops the detector from "remembering" a patient across folds.
- *CDC:* a fixed random sample of 50,000 survey responses keeps run times similar to Diabetes 130.
- The split keeps the **risk stratum** (older patients; people without access to care; the finance group chosen in notebook 01) at the same share and positive rate in all three parts.

*Taiwan credit (UCI 350, added Oct 10, 2026):* replaces South German Credit, whose 1,000 rows were too few for stable results. Undocumented education and marriage codes are merged into "other".

**Which datasets run:** `li.RUN_ONLY` in `li_lib.py`. With `RUN_ONLY = ("taiwancredit",)` only the Taiwan data is downloaded, cleaned and split; the Diabetes and CDC splits already in `data/` are left untouched, and the logs keep their rows.

**Inputs:** UCI downloads (automatic; or put the files in `data/` by hand).
**Outputs:** `data/{dataset}_train.csv`, `_val.csv`, `_test.csv`, `results/00_cleaning_log.csv`, `results/00_split_summary.csv`.

In [1]:
# --- setup: run this cell first -------------------------------------------
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path.cwd()))          # li_lib.py sits next to the notebooks
import li_lib as li
pd.set_option("display.width", 160, "display.max_columns", 30)
print("li_lib", li.VERSION, "| project folder:", li.ROOT)
li.versions()

li_lib 1.1.0 | project folder: D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity


,package,version
0,numpy,2.2.6
1,pandas,3.0.6
2,scikit-learn,1.9.1
3,xgboost,3.4.1
4,cleanlab,2.9.0
5,lingam,1.13.0
6,causal-learn,0.1.4.8


## 1. Download (or find) the raw files
If a download fails, open the UCI page in a browser, download the zip, and copy the file named below into the `data/` folder. Then rerun this cell.
- Diabetes 130: `diabetic_data.csv` from https://archive.ics.uci.edu/dataset/296
- CDC Diabetes Health Indicators: downloaded with `ucimlrepo` (UCI id 891) and saved as `cdc_diabetes_raw.csv`
- Taiwan credit-card default: downloaded with `ucimlrepo` (UCI id 350) and saved as `taiwan_credit_raw.csv`. **If that fails:** download https://archive.ics.uci.edu/static/public/350/data.csv, rename it `taiwan_credit_raw.csv` and put it in `data/` (the Excel file `default of credit card clients.xls` from the UCI page also works; the next cell finds it).

**Fallback for Diabetes 130:** if `diabetic_data.csv` cannot be downloaded, the cell uses your older `data/diabetes_130.csv`. That file has no patient IDs, so repeat patients cannot be removed; the notebook prints a warning, and the paper would need to say so. Get the raw file if at all possible.

In [2]:
RUN = li.run_configs()                      # set in li_lib.py: RUN_ONLY
print("running:", list(RUN))
raw_paths = {}
for cfg in RUN.values():
    try:
        raw_paths[cfg.name] = li.download_raw(cfg)
    except Exception as e:
        xls = li.DATA / "default of credit card clients.xls"
        if cfg.name == "taiwancredit" and xls.exists():
            raw_paths[cfg.name] = xls
            print("using the Excel file from the UCI page:", xls.name)
        else:
            print(f"!! {cfg.name}: {e}\n   -> download by hand (see the cell above) and rerun")
if "diabetes130" in RUN and "diabetes130" not in raw_paths and (li.DATA / "diabetes_130.csv").exists():
    raw_paths["diabetes130"] = li.DATA / "diabetes_130.csv"
    print("!! WARNING: using the older diabetes_130.csv (no patient IDs; repeat patients stay in)")
raw_paths

running: ['taiwancredit']
downloading UCI id 350 with ucimlrepo
saved D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\data\taiwan_credit_raw.csv


{'taiwancredit': WindowsPath('D:/Documents/PhD_program/DISSERTATION_RELATED/Dissertation_Studies_code/safety_notebooks/label_integrity/data/taiwan_credit_raw.csv')}

In [3]:
frames, logs = {}, []
for name, path in raw_paths.items():
    frames[name], lg = li.load_and_clean(li.CONFIGS[name], path)
    lg.insert(0, "dataset", name); logs.append(lg)

cleaning_log = li.save_merged(pd.concat(logs, ignore_index=True), "00_cleaning_log.csv", show=0)
cleaning_log[cleaning_log.dataset.isin(list(frames))]

kept 9 existing rows for ['cdcdiabetes', 'diabetes130', 'southgerman']
wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\00_cleaning_log.csv  (13 rows x 3 cols)


,dataset,step,rows_remaining
9,taiwancredit,raw clients,30000
10,taiwancredit,drop rows with missing values,30000
11,taiwancredit,drop exact duplicate rows,29965
12,taiwancredit,merge undocumented education / marriage codes ...,29965


## 3. Split 60 / 20 / 20 and save
**Look for:** the positive rate and the stratum share should be almost identical across train, validation and test.

In [4]:
rows = []
for name, df in frames.items():
    cfg = li.CONFIGS[name]
    parts = dict(zip(["train", "val", "test"], li.split_60_20_20(df, cfg)))
    for part, d in parts.items():
        d.to_csv(li.DATA / f"{name}_{part}.csv", index=False)
        s = li.stratum_mask(d, cfg)
        rows.append({"dataset": name, "part": part, "n": len(d),
                     "positive_rate": d[cfg.label].mean(),
                     "stratum": cfg.stratum_name, "stratum_share": s.mean(),
                     "stratum_positive_rate": d.loc[s, cfg.label].mean(),
                     "rest_positive_rate": d.loc[~s, cfg.label].mean()})
split_summary = pd.DataFrame(rows)
split_summary = li.save_merged(split_summary, "00_split_summary.csv", show=0)
split_summary.round(3)

kept 9 existing rows for ['cdcdiabetes', 'diabetes130', 'southgerman']
wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\00_split_summary.csv  (12 rows x 8 cols)


,dataset,part,n,positive_rate,stratum,stratum_share,stratum_positive_rate,rest_positive_rate
0,diabetes130,train,41992,0.090,age_80plus,0.184,0.106,0.086
1,diabetes130,val,13997,0.090,age_80plus,0.184,0.106,0.086
2,diabetes130,test,13998,0.090,age_80plus,0.184,0.106,0.086
3,cdcdiabetes,train,30000,0.157,no_care_access,0.126,0.157,0.157
4,cdcdiabetes,val,10000,0.156,no_care_access,0.126,0.157,0.156
5,cdcdiabetes,test,10000,0.157,no_care_access,0.126,0.157,0.156
6,southgerman,train,600,0.300,age_25_or_under,0.190,0.421,0.272
7,southgerman,val,200,0.300,age_25_or_under,0.190,0.421,0.272
8,southgerman,test,200,0.300,age_25_or_under,0.190,0.421,0.272
9,taiwancredit,train,17979,0.221,age_25_or_under,0.129,0.267,0.215


## What to look for before moving on
- [ ] Diabetes train has roughly 40,000+ patients; positive rate (readmitted within 30 days) is about 0.09-0.11.
- [ ] CDC train has 30,000 rows; positive rate (diabetes or prediabetes) is roughly 0.15; the no-care-access stratum is roughly 10-15%.
- [ ] Taiwan credit train has about 18,000 rows; positive rate (default next month) is about 0.22.
- [ ] Stratum shares match across the three parts.

**For the paper (Methods):** report the cleaning steps and final counts from `00_cleaning_log.csv`.